In [1]:
import glob
import os
import shutil
from pathlib import Path

import chardet


root = os.path.dirname(os.getcwd())
backup_root = os.path.join(root, ".codebackup", "root")
backup_root_src = os.path.join(backup_root, "src")

os.makedirs(backup_root, exist_ok = True)
os.makedirs(backup_root_src, exist_ok = True)

RED = "\033[91m"
RESET = "\033[0m"
GREEN = "\033[92m"
BLUE = "\033[94m"

In [3]:

import subprocess


source_folder = Path(os.path.join(root, "notebooks"))
dest_folder = Path(os.path.join(root, ".codebackup", "root", 'notebooks'))

print("From:", source_folder)
print("To:", dest_folder)

dest_folder.mkdir(parents = True, exist_ok = True)

notebooks = list(source_folder.glob("*.ipynb"))

for nb_path in notebooks:
    output_path = dest_folder / f"{nb_path.stem}.py"

    try:
        nb_rel = nb_path.relative_to(source_folder)
    except ValueError:
        nb_rel = nb_path

    try:
        out_rel = output_path.relative_to(dest_folder)
    except ValueError:
        out_rel = output_path

    print(f"Converting {nb_rel} -> {out_rel}")

    cmd = [
        "python",
        "-m", "jupytext",
        "--to", "py:percent",
        "--opt", "notebook_metadata_filter=-all",
        "--opt", "cell_metadata_filter=-all",
        "--output",
        str(output_path),
        str(nb_path),
    ]
    result = subprocess.run(
            cmd,
            text = True,
            capture_output = True
    )
    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)
    print("Exit code:", result.returncode)




From: C:\github\Tree-Canopy-Detection\notebooks
To: C:\github\Tree-Canopy-Detection\.codebackup\root\notebooks
Converting 00 colab setup.ipynb -> 00 colab setup.py
STDOUT:
 [jupytext] Reading C:\github\Tree-Canopy-Detection\notebooks\00 colab setup.ipynb in format ipynb
[jupytext] Updating the timestamp of 'C:\github\Tree-Canopy-Detection\.codebackup\root\notebooks\00 colab setup.py'

STDERR:
 C:\Users\johnh\AppData\Local\Programs\Python\Python313\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)

Exit code: 0
Converting 00 preflight check.ipynb -> 00 preflight check.py
STDOUT:
 [jupytext] Reading C:\github\Tree-Canopy-Detection\notebooks\00 preflight check.ipynb

In [ ]:
#  --- Copy py files ---
folders_to_copy = [root, os.path.join(root, "notebooks", ".codebackup/from_notebooks")]  # add more if needed

for folder in folders_to_copy:
    if os.path.isdir(folder):
        # Create a mirror backup folder under .codebackup/root
        relative_name = os.path.relpath(folder, root).replace(os.sep, "_")
        backup_target = os.path.join(backup_root, relative_name)
        os.makedirs(backup_target, exist_ok = True)

        py_files = glob.glob(os.path.join(folder, "*.py"))
        for fname in py_files:
            shutil.copy(fname, backup_target)
            print(f"Copied {RED}{fname} -> {backup_target}{RESET}\n")




In [ ]:
# # verify encoding of files
#
#
# def convert_to_utf8( full_path, encoding ):
#     """
#     Convert a file to UTF-8 if it's not already.
#     """
#     try:
#         with open(full_path, "r", encoding = encoding, errors = "replace") as infile:
#             content = infile.read()
#         with open(full_path, "w", encoding = "utf-8") as outfile:
#             outfile.write(content)
#         print(f"{GREEN}Converted {full_path} to UTF-8{RESET}")
#     except Exception as e:
#         print(f"{RED}Failed to convert {full_path}: {e}{RESET}")
#
#
# root = os.path.dirname(os.getcwd())
#
# spath = ""
# for dirpath, dirnames, filenames in os.walk(root):
#     for fname in filenames:
#
#         if fname.endswith((".py", ".ipynb", ".md", ".yaml", ".ini", ".json")):
#             if spath != dirpath:
#                 spath = dirpath
#                 print(f"\n----- {spath.replace(root + os.sep, '').upper()} -----")
#             full_path = os.path.join(dirpath, fname)
#
#             # detect encoding
#             with open(full_path, "rb") as rawfile:
#                 rawdata = rawfile.read()
#                 result = chardet.detect(rawdata)
#                 encoding = result["encoding"]
#                 confidence = result["confidence"]
#
#             rel_path = full_path.replace(root + os.sep, "")
#
#             if encoding and encoding.lower() != "utf-8":
#                 if encoding and encoding.lower() == "ascii":
#                     print(f"{rel_path} -> {BLUE}encoding = {encoding}{RESET}, confidence = {confidence:.2f}")
#                 else:
#                     print(f"{rel_path} -> {RED}encoding = {encoding}{RESET}, confidence = {confidence:.2f}")
#                     #convert_to_utf8(full_path, encoding)
#             else:
#                 print(f"{rel_path} -> {GREEN}encoding = {encoding}{RESET}, confidence = {confidence:.2f}")


In [ ]:
#  --- merge py files into single file ---

folders_to_process = ["src"]

for folder in folders_to_process:
    source_folder = os.path.join(root, folder)
    backup_root_folder = os.path.join(backup_root, folder)
    os.makedirs(backup_root_folder, exist_ok = True)

    for subfolder in os.listdir(source_folder):
        subfolder_path = os.path.join(source_folder, subfolder)

        if os.path.isdir(subfolder_path):
            py_files = glob.glob(os.path.join(subfolder_path, "*.py"))

            if py_files:
                output_file = os.path.join(backup_root_folder, f"_{subfolder}.py")

                with open(output_file, "w", encoding = "utf-8") as outfile:
                    for fname in py_files:
                        print(fname)

                        # detect encoding
                        with open(fname, "rb") as rawfile:
                            rawdata = rawfile.read()
                            result = chardet.detect(rawdata)
                            encoding = result["encoding"] or "utf-8"

                        # read with detected encoding, fallback to utf-8
                        with open(fname, "r", encoding = encoding, errors = "replace") as infile:
                            outfile.write(f"# From {fname}\n")
                            outfile.write(infile.read())
                            outfile.write("\n\n")

                print(f"Created {BLUE}{output_file}{RESET}\n")



In [ ]:
print("Done")